# <font color="steelblue">Satisfacción aerolínea</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.


## <font color="steelblue">Objetivos del proyecto</font>

A partir de una encuesta de satisfacción, construir, comparar y **desplegar** un clasificador que prediga si un pasajero queda **satisfecho** o **neutral/insatisfecho**. A diferencia de los proyectos clínicos, aquí el objetivo último es de **negocio**: entender **qué aspectos del servicio priorizar**. El proyecto entrena tres competencias propias:

* **Interpretabilidad accionable:** traducir las importancias del modelo (SHAP) en **recomendaciones** de inversión, con la cautela de **correlación ≠ causalidad**.
* **Disciplina metodológica con una partición dada:** el dataset ya viene dividido en *train*/*test*; hay que **respetarla** (no re-partir a capricho).
* **Semántica de los datos:** las 14 valoraciones usan **0 = "no aplicable"** (un **faltante**, no la peor nota).

Aplicaréis además el flujo completo: comparación de modelos, equilibrado, optimización, combinación, evaluación y despliegue.


## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

El conjunto recoge los resultados de una **encuesta de satisfacción** realizada por una aerolínea a sus pasajeros, publicada en Kaggle por *teejmahal20*. Contiene **~129.880 registros** y viene **pre-dividido** en `train.csv` (103.904 filas) y `test.csv` (25.976), una partición aproximada del 80/20 que el propio autor proporciona ya hecha.

Cada fila corresponde a **un pasajero y un vuelo**, y combina tres tipos de información de naturaleza muy distinta: quién es el pasajero, cómo fue el vuelo (datos objetivos, medidos) y **cómo valoró catorce aspectos del servicio** (datos subjetivos, declarados). Esa mezcla de lo objetivo y lo subjetivo es la clave para entender qué se puede y qué no se puede concluir con estos datos.

### <font color="steelblue">Diccionario de variables</font>

**Bloque 1 — Perfil del pasajero**

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `Gender` | Nominal | Female / Male | **Sexo** del pasajero. |
| `Customer Type` | Nominal | Loyal / disloyal | Si el pasajero pertenece al **programa de fidelización** de la aerolínea o es un cliente ocasional. |
| `Age` | Numérica | años | **Edad** del pasajero. |
| `Type of Travel` | Nominal | Personal / Business | **Motivo del viaje**: ocio/personal o negocios. Determina expectativas muy distintas sobre el servicio. |

**Bloque 2 — Características del vuelo** *(datos objetivos)*

| Variable | Tipo | Descripción |
|---|---|---|
| `Class` | **Ordinal** | **Clase del billete**: Business > Eco Plus > Eco. Está fuertemente asociada a `Type of Travel`. |
| `Flight Distance` | Numérica | **Distancia del vuelo**. Distingue trayectos cortos de largo radio, donde el confort y el entretenimiento pesan mucho más. |
| `Departure Delay in Minutes` | Numérica | **Retraso en la salida**. Distribución muy asimétrica: la mayoría son 0 y hay una cola larga de retrasos extremos. |
| `Arrival Delay in Minutes` | Numérica | **Retraso en la llegada**. Muy correlacionado con el de salida (~0,96) y con **algunos faltantes** (unos 310 en `train`). |

**Bloque 3 — Las 14 valoraciones de servicio** *(escala 0–5; datos subjetivos)*

Cubren el recorrido completo del pasajero, desde la reserva hasta la recogida del equipaje:

| Etapa | Variables |
|---|---|
| **Antes del vuelo** | `Ease of Online booking` (facilidad de reserva en línea), `Departure/Arrival time convenient` (conveniencia del horario), `Online boarding` (embarque en línea), `Checkin service` (servicio de facturación), `Gate location` (ubicación de la puerta) |
| **Durante el vuelo** | `Inflight wifi service` (wifi), `Seat comfort` (confort del asiento), `Leg room service` (espacio para las piernas), `Food and drink` (comida y bebida), `Inflight entertainment` (entretenimiento), `On-board service` (servicio a bordo), `Inflight service` (servicio en vuelo), `Cleanliness` (limpieza) |
| **Después del vuelo** | `Baggage handling` (manejo del equipaje) |

> ⚠️ **El 0 significa "no aplicable"**, no "muy insatisfecho". Un pasajero que no usó el wifi marca 0, igual que quien no consumió comida a bordo. Es el aviso más importante del preprocesado.

**Variable objetivo**

| Variable | Valores | Descripción |
|---|---|---|
| `satisfaction` | `satisfied` / `neutral or dissatisfied` | **Satisfacción global** declarada por el pasajero. Reparto **bastante equilibrado**: alrededor del 45 % de pasajeros satisfechos. |

> **Limitación del dataset:** por anonimización **no** están la aerolínea, la aeronave, el aeropuerto, el precio ni la fecha; podrían influir en la satisfacción. Tenlo en cuenta al interpretar.
> **Columnas a descartar:** suele haber un índice (`Unnamed: 0`) y un `id` no predictivos.

### <font color="steelblue">Advertencias metodológicas</font>

1. **El `0` rompe la escala: no tratéis las valoraciones como ordinales sin más.** Si el 0 significa «no aplicable», entonces la escala **no es monótona**: un 0 no es peor que un 1, es *otra cosa*. Un modelo que la trate como numérica interpretará que «no usar el wifi» es la peor experiencia posible de wifi, lo cual es falso. Las salidas razonables son **convertir el 0 en `NaN`** (añadiendo un indicador binario de «no aplicable», porque esa ausencia **es informativa**: quien no usa el wifi quizá viaja en trayectos cortos) o **tratarlo como categoría propia** con *one-hot*. Comparad ambos tratamientos: la diferencia en rendimiento e interpretación es un resultado del proyecto.

2. **Circularidad: esto no es una predicción, es una explicación.** Las catorce valoraciones y la satisfacción global **se recogieron en la misma encuesta y en el mismo momento**. La satisfacción global es, en gran medida, un **resumen** de las valoraciones parciales. Predecirla a partir de ellas no permite anticipar nada: cuando disponemos de las respuestas, el pasajero ya ha declarado su satisfacción. Ahora bien, **esto no invalida el ejercicio**: la pregunta legítima no es «¿podré predecir la satisfacción?», sino **«¿qué dimensiones del servicio determinan la satisfacción global?»**, que es exactamente lo que la aerolínea quiere saber para decidir dónde invertir. El proyecto es de **explicabilidad** (SHAP) más que de predicción, y conviene enunciarlo así.

3. **La etiqueta colapsa dos estados distintos.** «Neutral» y «dissatisfied» se han fundido en una sola clase. Un pasajero indiferente no es un pasajero descontento, y desde el punto de vista del negocio la diferencia es enorme. Esa fusión **oculta información** y sitúa la frontera de decisión en un punto arbitrario de una escala que, en origen, era ordinal.

4. **La partición ya viene hecha: respetadla.** Es tentador concatenar `train` y `test` y volver a partir. Si lo hacéis, perdéis el único conjunto de evaluación **verdaderamente independiente** que tenéis. Lo correcto es usar `train.csv` para todo (exploración, validación cruzada, ajuste de hiperparámetros) y reservar `test.csv` para una **única evaluación final**. Comprobad además que ningún `id` aparece en ambos ficheros antes de descartarlo.

5. **Multicolinealidad entre los dos retrasos.** `Departure Delay` y `Arrival Delay` miden esencialmente el mismo suceso, con una correlación próxima a 0,96. No perjudica a los árboles, pero **reparte arbitrariamente su importancia**, de modo que ninguna de las dos parecerá relevante aunque el retraso sí lo sea. Una alternativa interesante es construir una variable derivada: el **retraso recuperado en vuelo** (`Departure − Arrival`), que sí aporta información nueva.

6. **Faltantes con estructura.** Los ausentes de `Arrival Delay` son pocos (~0,3 %) y probablemente aleatorios: una imputación simple basta. Pero los ceros de las valoraciones, si los convertís en `NaN`, **no son aleatorios en absoluto** —dependen de si el servicio estaba disponible o se usó—. Es el mismo fenómeno que hemos visto en otros conjuntos: una ausencia **estructural**, que debe codificarse, no imputarse a ciegas.

7. **Sesgos de la encuesta.** Los datos no proceden de todos los pasajeros, sino de **quienes respondieron**. Los muy satisfechos y los muy descontentos contestan más que los indiferentes (**sesgo de autoselección**), de modo que la prevalencia observada no es la de la población de pasajeros. Además, las valoraciones son **percepciones**, no mediciones: dos personas con la misma experiencia pueden puntuarla de forma distinta.

8. **Variables omitidas.** Tu aviso sobre la anonimización tiene una consecuencia técnica precisa: el **precio del billete**, la **hora del vuelo** y la **ruta** son variables ocultas correlacionadas con `Class`, `Flight Distance` y con la satisfacción. El modelo atribuirá a las variables observadas efectos que en realidad corresponden a las omitidas (**sesgo de variable omitida**), lo que refuerza la advertencia de no leer las importancias en clave causal: que `Online boarding` tenga un SHAP alto **no demuestra** que mejorar el embarque en línea aumente la satisfacción.

9. **Consideraciones éticas.** `Gender` y `Age` son **atributos sensibles**. Si el modelo se usara para priorizar la atención al cliente, conviene evaluar el rendimiento **por subgrupos** y no solo el global. Nótese además que la anonimización eliminó el resto de información demográfica, lo que **impide** hacer un análisis de equidad completo.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Respeta la partición dada.** Usa `train.csv` para **todo** (EDA, CV, ajuste de hiperparámetros) y `test.csv` **solo** para la evaluación final. **No** concatenes y re-partas salvo que lo justifiques explícitamente.
2. **Trata el 0 de las valoraciones como faltante** (no como nota mínima): decídelo y documéntalo (imputar o categoría "no aplica").
3. **Sin fuga en el preprocesado:** imputación, codificación, escalado y remuestreo dentro de un **`Pipeline`**, ajustados **solo con el *train*** (y en cada pliegue de la CV).
4. **Equilibrado solo en *train***; aquí el reparto es parejo, así que su efecto será pequeño (resultado válido).
5. **Interpreta para actuar, con cautela:** una alta importancia **no** implica causalidad; sé prudente al recomendar inversiones.
6. **Reproducibilidad y honestidad:** `random_state` fijado; reporta lo que no funcionó y las limitaciones.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

> **Importante:** el dataset **ya viene dividido**. Cargamos `train` y `test` **por separado** y **mantenemos** esa partición (a diferencia de concatenarlos).

In [ ]:
# !pip -q install kagglehub imbalanced-learn scikit-learn shap gradio
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
print("Ruta:", path, "| Archivos:", os.listdir(path))

df_train = pd.read_csv(os.path.join(path, 'train.csv'))
df_test  = pd.read_csv(os.path.join(path, 'test.csv'))
print(f"Train: {df_train.shape[0]:,} × {df_train.shape[1]}   Test: {df_test.shape[0]:,} × {df_test.shape[1]}")

# Mantenemos la partición oficial. (Concatenar y re-partir descarta el split dado: NO recomendado.)
df_train.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA</font>

> Haced el EDA **sobre `train`** (mirar el *test* es fuga).

**Tareas obligatorias**
1. **Objetivo.** Distribución de `satisfaction` (confirmad el reparto **equilibrado**).
2. **Tipos y columnas a descartar** (`Unnamed: 0`, `id`). Identificad las **categóricas** (`Gender`, `Customer Type`, `Type of Travel`), la **ordinal** (`Class`), las **numéricas** (edad, distancia, retrasos) y las **14 valoraciones**.
3. **Semántica del 0 (clave).** Comprobad cuántos **0 = "no aplicable"** hay en cada valoración y decidid cómo tratarlos.
4. **Faltantes.** `Arrival Delay in Minutes` (y los 0 que consideréis faltantes); plan de imputación.
5. **Multicolinealidad.** Correlación entre `Departure Delay` y `Arrival Delay` y entre valoraciones.
6. **Relación con el objetivo.** Satisfacción por `Class`, `Type of Travel`, `Online boarding`, `Inflight wifi service`… (la literatura señala online boarding y wifi como top).
7. **Conclusión:** 3–4 hallazgos **con lectura de negocio**.

> **A responder:** ¿por qué tratar el 0 como "nota mínima" distorsionaría el análisis?

# <font color="steelblue">Fase 2 — Preprocesado (respetando la partición)</font>

**Tareas obligatorias**
1. **Descarta** `Unnamed: 0` e `id`. Separa `X`/`y` en **`train`** y en **`test`** por separado (`y` = `satisfaction` → 1 = satisfied, 0 = neutral/dissatisfied).
2. **0 = "no aplicable" → faltante:** en las 14 valoraciones, sustituye 0 por `NaN` (o crea indicadores) y **decide la imputación**, dentro del `Pipeline`.
3. **Codificación:** binarias (`Gender`, `Customer Type`, `Type of Travel`) a 0/1; **`Class` como ordinal** (Eco < Eco Plus < Business). **`Age`/distancia/retrasos** escalados para logística/SVM/kNN (los árboles no).
4. **`Pipeline`/`ColumnTransformer`** con todo el preprocesado, **ajustado solo con `train`**.
5. **No** toques el *test* hasta la Fase 7.

> **A responder:** ¿qué pasaría si imputaras el faltante de `Arrival Delay` usando la media de *train*+*test* juntos?

# <font color="steelblue">Fase 3 — Modelos base y comparación</font>

**Tareas obligatorias** (CV **sobre `train`**)
1. Comparad **≥5 familias** del curso: **Regresión logística**, **kNN**, **SVM**, **Árbol**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost — suelen ganar aquí), **Naive Bayes**.
2. **CV estratificada** sobre `train`; con el reparto equilibrado, **`accuracy`/`f1`/`roc_auc`** son razonables (mirad varias).
3. **Tabla** comparativa y comentario.

> **Cómputo:** ~104k filas en *train*; para explorar podéis **submuestrear** y reentrenar el ganador con todo. Documentadlo.

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

El reparto es **bastante equilibrado**, así que el remuestreo aportará poco; aun así es obligatorio **medirlo** (material **11**) sobre los 2–3 mejores modelos:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'`.
3. (Opcional) **SMOTE**/submuestreo, para confirmar que con clases parejas aportan poco.

Reportad **F1**, **recall** por clase y **ROC-AUC**, y comentad si **aporta**.

> **Sin fugas:** remuestreo dentro de `Pipeline` de *imbalanced-learn*, solo en `train`.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

1. Optimizad los **2–3 mejores** (modelo + equilibrado).
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada **sobre `train`** y la métrica elegida; búsqueda **sobre el `Pipeline`** (prefijo `clf__`).
3. Con ~104k filas, `RandomizedSearchCV`/Optuna sobre **submuestra** es lo eficiente; reentrenad el ganador con todo `train`.
4. Reportad mejores hiperparámetros y la mejora.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual**: ¿mejora? ¿compensa el coste y la **menor interpretabilidad** (importante porque el objetivo es explicar)?
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación e interpretabilidad accionable</font>

Esta es la fase **distintiva**: el modelo es un medio para **decidir dónde invertir**. Se evalúa en el **`test` proporcionado**, una sola vez.

**Tareas obligatorias**
1. **Métricas finales** en `test`: **matriz de confusión**, `classification_report`, **F1**, **ROC-AUC**.
2. **Importancia de variables (clave).** Con `permutation_importance` y/o **SHAP**, ordenad los factores. ¿Aparecen **`Online boarding`**, **`Inflight wifi service`**, **`Class`**, **`Type of Travel`**, comodidad del asiento…, como en la literatura?
3. **Recomendaciones de negocio.** Traducid las 3–5 variables más influyentes en **acciones priorizadas** ("mejorar el embarque online", "wifi a bordo"…). Apoyaos en **dependencia parcial (PDP)** o gráficos SHAP para ver el sentido del efecto.
4. **Correlación ≠ causalidad (obligatorio).** Advertid que una alta importancia **no garantiza** que mejorar ese aspecto **cause** más satisfacción (posibles confusores; faltan precio/aerolínea/ruta). Proponed cómo se validaría (p. ej. análisis causal o un experimento).
5. **Discusión crítica:** limitaciones del dataset y validez de las recomendaciones.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

1. **Persistencia:** guardad el **`Pipeline` completo** con `joblib`.
2. **Función de predicción:** `predecir_satisfaccion(...)` que devuelva la clase y la **probabilidad**.
3. **Interfaz / simulador "what-if":** app **Gradio** (o `ipywidgets`) con deslizadores para las valoraciones y selectores para clase/tipo de viaje; al cambiar, por ejemplo, `Online boarding`, se ve cómo cambia la probabilidad de satisfacción — útil para **explorar palancas**. En Colab da un **enlace público** (incluidlo).
4. (Opcional, nota extra) **Streamlit**/**FastAPI**.

> **Aviso:** herramienta **educativa**; las relaciones son **predictivas**, no necesariamente **causales**.

# <font color="steelblue">Pistas y errores típicos</font>

* **Respeta el split.** CV y ajuste sobre `train`; `test` solo al final. Concatenar y re-partir descarta la partición oficial.
* **0 = "no aplicable".** En las valoraciones, el 0 no es la peor nota: trátalo como **faltante** (imputar o categoría aparte) y justifícalo.
* **Multicolinealidad de los retrasos.** `Departure` y `Arrival Delay` van casi de la mano: afecta a modelos lineales (considera quitar uno o regularizar).
* **Interpretar para actuar, con humildad.** Una variable muy importante no es necesariamente una **palanca causal**; faltan precio/aerolínea/ruta. Propón cómo validarlo.
* **Despliegue:** guarda el **Pipeline entero**; el simulador *what-if* es ideal para este caso de negocio.

# <font color="steelblue">Referencias</font>

* Mahal, T. J. (2020). *Airline Passenger Satisfaction*. Kaggle.
* *Enhancing Airline Customer Satisfaction: A Machine Learning and Causal Analysis Approach*. arXiv, 2024.
* *Predictive Models with XAI: A Comparative Study of Enhancing Airline Customer Satisfaction*. ACM, 2023.
* *Investigating airline passenger satisfaction: Data mining method*. Transportation Research Part E, 2021.
* Cuadernos del curso: *Boosting*, *Random Forest*, *Regresión logística binaria*, *Equilibrando las muestras*.